# Question Difficulty & Item Quality Analysis

This notebook performs per-question analysis to identify questions that models **consistently fail**,
which may indicate:

- Poorly worded questions
- Incorrect answer keys
- Ambiguous or under-specified questions
- Questions outside the benchmark's intended scope

## Analyses

1. **MCQ Per-Question Failure Rates** — which questions do most/all models get wrong?
2. **Wrong Answer Consensus** — do models agree on a specific wrong answer? (bad answer key signal)
3. **OSQ Per-Question Scores** — which questions get low OSQ scores across all models?
4. **Item Discrimination** — classical test theory metrics to flag problematic items
5. **Category Concentration** — are failing questions concentrated in specific INCOSE topics?
6. **Summary & Export** — consolidated flagged-questions table for manual review

---
## 1. Setup & Data Loading

### 1.1 Imports

In [ ]:
# Standard libraries
import pandas as pd
import numpy as np
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistics
from scipy import stats
from scipy.stats import (
    chi2_contingency,
    pearsonr, spearmanr,
    pointbiserialr
)

# Progress
from tqdm.auto import tqdm

# Local parsers
import sys
from parsers import (
    parse_mcq_samples,
    parse_osq_judged_samples,
    align_mcq_osq_results,
    filter_osq_data,
    get_available_judges_and_prompts
)

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print('Imports complete')

### 1.2 Configuration

In [ ]:
# Paths
phase4_dir = Path('../phase4_inference/output')
phase5_dir = Path('../phase5_llm_as_a_judge')
output_dir = Path('output_v2')
output_dir.mkdir(exist_ok=True)

# Thresholds for flagging
FAILURE_RATE_THRESHOLD = 0.75      # Flag questions with >= 75% failure rate
CONSENSUS_WRONG_THRESHOLD = 0.50   # Flag if >= 50% of models pick the same wrong answer
LOW_OSQ_SCORE_THRESHOLD = 40       # Flag questions with mean OSQ score < 40/100
LOW_DISCRIMINATION_THRESHOLD = 0.10  # Flag items with discrimination index < 0.10
TOP_N_HEATMAP = 50                 # Number of hardest questions to show in heatmap

# Position variants for MCQ
POSITION_VARIANTS = ['a', 'b', 'c', 'd']

print(f'Configuration complete')
print(f'  Output directory: {output_dir.absolute()}')
print(f'  Failure rate threshold: {FAILURE_RATE_THRESHOLD}')
print(f'  Consensus wrong threshold: {CONSENSUS_WRONG_THRESHOLD}')
print(f'  Low OSQ score threshold: {LOW_OSQ_SCORE_THRESHOLD}')
print(f'  Low discrimination threshold: {LOW_DISCRIMINATION_THRESHOLD}')

### 1.3 Load MCQ Data

In [ ]:
print('Loading MCQ samples...')
mcq_df = pd.DataFrame(parse_mcq_samples(phase4_dir, use_latest=True))

print(f'  Loaded {len(mcq_df)} MCQ samples')
print(f'  Models: {mcq_df["model"].nunique()}')
print(f'  Questions: {mcq_df["question_id"].nunique()}')
print(f'  Variants: {sorted(mcq_df["variant"].unique())}')
print(f'\nModels in dataset:')
for m in sorted(mcq_df['model'].unique()):
    print(f'  - {m}')

### 1.4 Load OSQ Data

In [ ]:
print('Loading OSQ judged samples...')
osq_df = pd.DataFrame(parse_osq_judged_samples(phase5_dir))

print(f'  Loaded {len(osq_df)} OSQ samples')
print(f'  Models: {osq_df["model"].nunique()}')
print(f'  Questions: {osq_df["question_id"].nunique()}')
print(f'  Judges: {sorted(osq_df["judge_model"].unique())}')

# Filter to successfully scored samples
osq_scored = osq_df[osq_df['parse_status'] == 'success'].copy()
print(f'\n  Successfully scored OSQ samples: {len(osq_scored)}')

---
## 2. MCQ Per-Question Failure Analysis

### 2.1 Per-Question Failure Rates

In [ ]:
# Use fixed-position variants (a, b, c, d) for consistent failure analysis
mcq_fixed = mcq_df[mcq_df['variant'].isin(POSITION_VARIANTS)].copy()

# Compute per-question, per-model average correctness across position variants
mcq_question_model = mcq_fixed.groupby(['question_id', 'model']).agg(
    mean_correct=('is_correct', 'mean'),
    n_variants=('variant', 'nunique')
).reset_index()

# Compute per-question failure rate across all models
question_stats = mcq_question_model.groupby('question_id').agg(
    n_models=('model', 'nunique'),
    mean_accuracy=('mean_correct', 'mean'),
).reset_index()
question_stats['failure_rate'] = 1 - question_stats['mean_accuracy']

# Add question metadata
question_meta = mcq_fixed.drop_duplicates('question_id')[['question_id', 'category', 'tags', 'question', 'correct_answer']]
question_stats = question_stats.merge(question_meta, on='question_id', how='left')

# Sort by failure rate
question_stats = question_stats.sort_values('failure_rate', ascending=False).reset_index(drop=True)

print(f'Per-question failure rates computed for {len(question_stats)} questions')
print(f'\nFailure rate distribution:')
print(question_stats['failure_rate'].describe())

n_hard = (question_stats['failure_rate'] >= FAILURE_RATE_THRESHOLD).sum()
print(f'\nQuestions with failure rate >= {FAILURE_RATE_THRESHOLD}: {n_hard}')

### 2.2 Failure Rate Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of failure rates
axes[0].hist(question_stats['failure_rate'], bins=30, edgecolor='black', alpha=0.7, color='steelblue')
axes[0].axvline(x=FAILURE_RATE_THRESHOLD, color='red', linestyle='--', linewidth=2,
                label=f'Threshold ({FAILURE_RATE_THRESHOLD})')
axes[0].set_xlabel('Failure Rate (across all models)')
axes[0].set_ylabel('Number of Questions')
axes[0].set_title('Distribution of MCQ Question Failure Rates')
axes[0].legend()

# Cumulative distribution
sorted_rates = np.sort(question_stats['failure_rate'])
cdf = np.arange(1, len(sorted_rates) + 1) / len(sorted_rates)
axes[1].plot(sorted_rates, cdf, color='steelblue', linewidth=2)
axes[1].axvline(x=FAILURE_RATE_THRESHOLD, color='red', linestyle='--', linewidth=2,
                label=f'Threshold ({FAILURE_RATE_THRESHOLD})')
axes[1].set_xlabel('Failure Rate')
axes[1].set_ylabel('Cumulative Proportion')
axes[1].set_title('CDF of Question Failure Rates')
axes[1].legend()

plt.tight_layout()
plt.savefig(output_dir / 'fig_failure_rate_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: fig_failure_rate_distribution.png')

### 2.3 Heatmap: Hardest Questions x Models

In [ ]:
# Get the top-N hardest questions
hardest_qids = question_stats.head(TOP_N_HEATMAP)['question_id'].tolist()

# Pivot: question (rows) x model (cols), values = mean correctness across variants
heatmap_data = mcq_question_model[mcq_question_model['question_id'].isin(hardest_qids)].pivot_table(
    index='question_id',
    columns='model',
    values='mean_correct',
    fill_value=0
)

# Sort rows by overall difficulty (hardest at top)
row_order = question_stats[question_stats['question_id'].isin(hardest_qids)].sort_values(
    'failure_rate', ascending=False
)['question_id']
heatmap_data = heatmap_data.loc[row_order]

# Sort columns by overall model accuracy
model_accuracy = mcq_fixed.groupby('model')['is_correct'].mean().sort_values()
col_order = [m for m in model_accuracy.index if m in heatmap_data.columns]
heatmap_data = heatmap_data[col_order]

fig, ax = plt.subplots(figsize=(16, max(10, TOP_N_HEATMAP * 0.3)))
sns.heatmap(
    heatmap_data,
    cmap='RdYlGn',
    vmin=0, vmax=1,
    linewidths=0.5,
    cbar_kws={'label': 'Mean Accuracy (across position variants)'},
    ax=ax
)
ax.set_title(f'Top {TOP_N_HEATMAP} Hardest Questions x Models (sorted by difficulty)')
ax.set_ylabel('Question ID')
ax.set_xlabel('Model')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(output_dir / 'fig_hardest_questions_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: fig_hardest_questions_heatmap.png')

### 2.4 Position-Invariant Failure Analysis

In [ ]:
# For each question, check if it fails across ALL position variants for each model
# A question is "position-invariant failed" if a model gets it wrong in all 4 positions

pos_analysis = mcq_fixed.pivot_table(
    index=['question_id', 'model'],
    columns='variant',
    values='is_correct',
    fill_value=np.nan
)

# Check if model fails all 4 variants for each question
pos_analysis['all_wrong'] = (pos_analysis[POSITION_VARIANTS] == False).all(axis=1)
pos_analysis['all_correct'] = (pos_analysis[POSITION_VARIANTS] == True).all(axis=1)
pos_analysis['n_correct'] = pos_analysis[POSITION_VARIANTS].sum(axis=1)

# Per-question: what fraction of models fail ALL variants?
pos_invariant = pos_analysis.groupby('question_id').agg(
    n_models=('all_wrong', 'count'),
    n_all_wrong=('all_wrong', 'sum'),
    n_all_correct=('all_correct', 'sum'),
).reset_index()
pos_invariant['pct_all_wrong'] = pos_invariant['n_all_wrong'] / pos_invariant['n_models']
pos_invariant['pct_all_correct'] = pos_invariant['n_all_correct'] / pos_invariant['n_models']

pos_invariant = pos_invariant.sort_values('pct_all_wrong', ascending=False)

n_pos_invariant_fail = (pos_invariant['pct_all_wrong'] >= FAILURE_RATE_THRESHOLD).sum()
print(f'Questions where >= {FAILURE_RATE_THRESHOLD:.0%} of models fail ALL 4 positions: {n_pos_invariant_fail}')
print(f'\nTop 20 position-invariant failures:')
display(pos_invariant.head(20))

### 2.5 Table: High-Failure Questions

In [ ]:
# Questions with failure rate >= threshold
high_failure = question_stats[question_stats['failure_rate'] >= FAILURE_RATE_THRESHOLD].copy()

# Add position-invariant info
high_failure = high_failure.merge(
    pos_invariant[['question_id', 'pct_all_wrong']],
    on='question_id',
    how='left'
)

print(f'\nHigh-Failure Questions (failure rate >= {FAILURE_RATE_THRESHOLD}):')
print(f'Total: {len(high_failure)}')

display_cols = ['question_id', 'failure_rate', 'pct_all_wrong', 'n_models', 'category', 'correct_answer', 'question']
display(high_failure[display_cols].head(30))

---
## 3. Wrong Answer Consensus Analysis

### 3.1 What Wrong Answer Do Models Choose?

In [ ]:
# For high-failure questions, analyze the distribution of wrong answers
high_failure_qids = high_failure['question_id'].tolist()

# Get all incorrect responses for these questions (use variant 'a' as canonical, or all variants)
wrong_answers = mcq_fixed[
    (mcq_fixed['question_id'].isin(high_failure_qids)) &
    (~mcq_fixed['is_correct'])
].copy()

print(f'Analyzing wrong answers for {len(high_failure_qids)} high-failure questions')
print(f'Total incorrect responses: {len(wrong_answers)}')

### 3.2 Consensus Wrong Answer Detection

In [ ]:
# For each high-failure question, find the most common wrong answer
consensus_results = []

for qid in high_failure_qids:
    q_wrong = wrong_answers[wrong_answers['question_id'] == qid]
    q_all = mcq_fixed[mcq_fixed['question_id'] == qid]
    
    if len(q_wrong) == 0:
        continue
    
    # Count wrong answer choices
    wrong_counts = q_wrong['model_answer'].value_counts()
    most_common_wrong = wrong_counts.index[0]
    most_common_wrong_count = wrong_counts.iloc[0]
    total_responses = len(q_all)
    
    # Consensus rate = fraction of ALL responses that chose this wrong answer
    consensus_rate = most_common_wrong_count / total_responses if total_responses > 0 else 0
    
    # Get the correct answer for comparison
    correct = q_all['correct_answer'].iloc[0]
    
    consensus_results.append({
        'question_id': qid,
        'correct_answer': correct,
        'consensus_wrong_answer': most_common_wrong,
        'consensus_wrong_count': most_common_wrong_count,
        'total_responses': total_responses,
        'consensus_rate': consensus_rate,
        'n_distinct_wrong_answers': len(wrong_counts)
    })

consensus_df = pd.DataFrame(consensus_results).sort_values('consensus_rate', ascending=False)

print(f'Consensus analysis complete for {len(consensus_df)} questions')
print(f'\nConsensus rate distribution:')
print(consensus_df['consensus_rate'].describe())

### 3.3 Flagged: Potential Answer Key Errors

In [ ]:
# Flag questions where a majority agrees on a wrong answer
consensus_flagged = consensus_df[consensus_df['consensus_rate'] >= CONSENSUS_WRONG_THRESHOLD].copy()

# Add question text
consensus_flagged = consensus_flagged.merge(
    question_meta[['question_id', 'question', 'category']],
    on='question_id',
    how='left'
)

print(f'Questions with consensus wrong answer rate >= {CONSENSUS_WRONG_THRESHOLD}:')
print(f'Total flagged: {len(consensus_flagged)}')
print(f'\nThese questions may have an INCORRECT answer key.')
print(f'Models agree on a different answer than the one marked correct.')

if len(consensus_flagged) > 0:
    display(consensus_flagged[[
        'question_id', 'correct_answer', 'consensus_wrong_answer',
        'consensus_rate', 'total_responses', 'category', 'question'
    ]])
else:
    print('  No questions flagged at this threshold.')

---
## 4. OSQ Per-Question Score Analysis

### 4.1 Per-Question Mean OSQ Scores

In [ ]:
# Compute per-question mean OSQ score across all models and judges
osq_question_stats = osq_scored.groupby('question_id').agg(
    mean_total_score=('total_score', 'mean'),
    std_total_score=('total_score', 'std'),
    median_total_score=('total_score', 'median'),
    n_evaluations=('total_score', 'count'),
    mean_technical_accuracy=('technical_accuracy', 'mean'),
    mean_conceptual_understanding=('conceptual_understanding', 'mean'),
    mean_completeness=('completeness', 'mean'),
    mean_clarity_organization=('clarity_organization', 'mean'),
    mean_professional_relevance=('professional_relevance', 'mean'),
).reset_index()

# Add metadata
osq_meta = osq_scored.drop_duplicates('question_id')[['question_id', 'category', 'tags', 'blooms_level']]
osq_question_stats = osq_question_stats.merge(osq_meta, on='question_id', how='left')

osq_question_stats = osq_question_stats.sort_values('mean_total_score').reset_index(drop=True)

print(f'OSQ per-question stats computed for {len(osq_question_stats)} questions')
print(f'\nMean total score distribution:')
print(osq_question_stats['mean_total_score'].describe())

n_low_osq = (osq_question_stats['mean_total_score'] < LOW_OSQ_SCORE_THRESHOLD).sum()
print(f'\nQuestions with mean OSQ score < {LOW_OSQ_SCORE_THRESHOLD}: {n_low_osq}')

### 4.2 OSQ Score Distribution Plot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(osq_question_stats['mean_total_score'], bins=30, edgecolor='black', alpha=0.7, color='darkorange')
axes[0].axvline(x=LOW_OSQ_SCORE_THRESHOLD, color='red', linestyle='--', linewidth=2,
                label=f'Threshold ({LOW_OSQ_SCORE_THRESHOLD})')
axes[0].set_xlabel('Mean OSQ Total Score (per question)')
axes[0].set_ylabel('Number of Questions')
axes[0].set_title('Distribution of Per-Question Mean OSQ Scores')
axes[0].legend()

# Rubric dimension breakdown for low-scoring questions
low_osq = osq_question_stats[osq_question_stats['mean_total_score'] < LOW_OSQ_SCORE_THRESHOLD]
rubric_dims = ['mean_technical_accuracy', 'mean_conceptual_understanding',
               'mean_completeness', 'mean_clarity_organization', 'mean_professional_relevance']
dim_labels = ['Technical\nAccuracy', 'Conceptual\nUnderstanding', 'Completeness',
              'Clarity &\nOrganization', 'Professional\nRelevance']

if len(low_osq) > 0:
    dim_means = [low_osq[d].mean() for d in rubric_dims]
    axes[1].bar(dim_labels, dim_means, color='darkorange', alpha=0.7, edgecolor='black')
    axes[1].set_ylabel('Mean Score (out of 20)')
    axes[1].set_title(f'Rubric Breakdown: Low-Scoring Questions (n={len(low_osq)})')
    axes[1].set_ylim(0, 20)
else:
    axes[1].text(0.5, 0.5, 'No low-scoring questions found', ha='center', va='center',
                 transform=axes[1].transAxes)

plt.tight_layout()
plt.savefig(output_dir / 'fig_osq_question_scores.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: fig_osq_question_scores.png')

### 4.3 MCQ Failure Rate vs OSQ Score (Cross-Format)

In [ ]:
# Merge MCQ failure rates with OSQ question scores
cross_format = question_stats[['question_id', 'failure_rate', 'category']].merge(
    osq_question_stats[['question_id', 'mean_total_score']],
    on='question_id',
    how='inner'
)

print(f'Matched questions for cross-format analysis: {len(cross_format)}')

if len(cross_format) > 0:
    fig, ax = plt.subplots(figsize=(10, 8))
    
    scatter = ax.scatter(
        cross_format['failure_rate'],
        cross_format['mean_total_score'],
        alpha=0.5,
        s=30,
        c='steelblue',
        edgecolors='white',
        linewidth=0.5
    )
    
    # Highlight problem zone
    ax.axvline(x=FAILURE_RATE_THRESHOLD, color='red', linestyle='--', alpha=0.5, label='MCQ failure threshold')
    ax.axhline(y=LOW_OSQ_SCORE_THRESHOLD, color='orange', linestyle='--', alpha=0.5, label='OSQ score threshold')
    
    # Shade the "bad question" zone
    ax.axvspan(FAILURE_RATE_THRESHOLD, 1.0, alpha=0.05, color='red')
    ax.axhspan(0, LOW_OSQ_SCORE_THRESHOLD, alpha=0.05, color='orange')
    
    # Label the quadrants
    ax.text(0.05, 95, 'Easy (both formats)', fontsize=9, alpha=0.6)
    ax.text(0.80, 95, 'MCQ-hard, OSQ-easy\n(format-specific)', fontsize=9, alpha=0.6, ha='center')
    ax.text(0.05, 10, 'MCQ-easy, OSQ-hard\n(format-specific)', fontsize=9, alpha=0.6)
    ax.text(0.80, 10, 'Hard (both formats)\nReview candidates', fontsize=9, alpha=0.6,
            ha='center', color='red', fontweight='bold')
    
    # Correlation
    r, p = spearmanr(cross_format['failure_rate'], cross_format['mean_total_score'])
    ax.set_title(f'MCQ Failure Rate vs. Mean OSQ Score (Spearman r={r:.3f}, p={p:.2e})')
    ax.set_xlabel('MCQ Failure Rate (higher = harder)')
    ax.set_ylabel('Mean OSQ Total Score (lower = harder)')
    ax.legend()
    
    plt.tight_layout()
    plt.savefig(output_dir / 'fig_mcq_vs_osq_question.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: fig_mcq_vs_osq_question.png')
    
    # Count questions in the "bad" quadrant
    both_bad = cross_format[
        (cross_format['failure_rate'] >= FAILURE_RATE_THRESHOLD) &
        (cross_format['mean_total_score'] < LOW_OSQ_SCORE_THRESHOLD)
    ]
    print(f'\nQuestions failing in BOTH formats: {len(both_bad)}')
else:
    print('No matched questions found between MCQ and OSQ.')

---
## 5. Item Discrimination Analysis

### 5.1 Classical Item Discrimination Index

The **discrimination index** measures how well a question differentiates between high-performing and low-performing models.
A good question should be answered correctly by high-performers and incorrectly by low-performers.

- **Point-biserial correlation**: correlation between item correctness (0/1) and total test score
- Items with low or negative discrimination may be problematic

In [ ]:
# Use a single representative variant (e.g., 'a') for discrimination analysis
# to avoid position bias confounds
mcq_variant_a = mcq_fixed[mcq_fixed['variant'] == 'a'].copy()

# Create question x model correctness matrix
correctness_matrix = mcq_variant_a.pivot_table(
    index='model',
    columns='question_id',
    values='is_correct',
    fill_value=0
).astype(int)

# Total score per model (sum of correct answers)
model_total_scores = correctness_matrix.sum(axis=1)

# Compute point-biserial correlation for each question
discrimination_results = []

for qid in correctness_matrix.columns:
    item_scores = correctness_matrix[qid]
    
    # Skip if no variance (all correct or all wrong)
    if item_scores.std() == 0:
        discrimination_results.append({
            'question_id': qid,
            'difficulty': item_scores.mean(),
            'discrimination': 0.0,
            'rpb_pvalue': 1.0,
            'note': 'no_variance'
        })
        continue
    
    rpb, pval = pointbiserialr(item_scores, model_total_scores)
    
    discrimination_results.append({
        'question_id': qid,
        'difficulty': item_scores.mean(),  # P-value (proportion correct)
        'discrimination': rpb,
        'rpb_pvalue': pval,
        'note': 'ok'
    })

disc_df = pd.DataFrame(discrimination_results)

print(f'Item discrimination computed for {len(disc_df)} questions')
print(f'\nDiscrimination index distribution:')
print(disc_df['discrimination'].describe())

n_low_disc = (disc_df['discrimination'] < LOW_DISCRIMINATION_THRESHOLD).sum()
n_negative = (disc_df['discrimination'] < 0).sum()
print(f'\nItems with discrimination < {LOW_DISCRIMINATION_THRESHOLD}: {n_low_disc}')
print(f'Items with NEGATIVE discrimination: {n_negative}')

### 5.2 Difficulty vs. Discrimination Plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

# Color by quality zone
colors = []
for _, row in disc_df.iterrows():
    if row['discrimination'] < 0:
        colors.append('red')       # Negative discrimination
    elif row['discrimination'] < LOW_DISCRIMINATION_THRESHOLD:
        colors.append('orange')    # Low discrimination
    else:
        colors.append('green')     # Good discrimination

ax.scatter(disc_df['difficulty'], disc_df['discrimination'],
           c=colors, alpha=0.6, s=30, edgecolors='white', linewidth=0.5)

# Reference lines
ax.axhline(y=0, color='red', linestyle='-', alpha=0.3)
ax.axhline(y=LOW_DISCRIMINATION_THRESHOLD, color='orange', linestyle='--', alpha=0.3,
           label=f'Low discrimination ({LOW_DISCRIMINATION_THRESHOLD})')

# Labels
ax.set_xlabel('Difficulty (P = proportion correct)')
ax.set_ylabel('Discrimination (point-biserial r)')
ax.set_title('Item Difficulty vs. Discrimination')
ax.legend()

# Add count annotations
n_good = sum(1 for c in colors if c == 'green')
n_low = sum(1 for c in colors if c == 'orange')
n_neg = sum(1 for c in colors if c == 'red')
ax.text(0.02, 0.98, f'Good: {n_good}  |  Low: {n_low}  |  Negative: {n_neg}',
        transform=ax.transAxes, va='top', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig(output_dir / 'fig_difficulty_vs_discrimination.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: fig_difficulty_vs_discrimination.png')

---
## 6. Category Concentration of Failing Questions

In [ ]:
# How are high-failure questions distributed across INCOSE categories?
flagged_qids = set(high_failure['question_id'].tolist())

# All questions with category info
all_q_cats = question_stats[['question_id', 'category']].copy()
all_q_cats['is_flagged'] = all_q_cats['question_id'].isin(flagged_qids)

# Count flagged vs total per category
cat_counts = all_q_cats.groupby('category').agg(
    total=('question_id', 'count'),
    flagged=('is_flagged', 'sum')
).reset_index()
cat_counts['flagged_pct'] = (cat_counts['flagged'] / cat_counts['total'] * 100)
cat_counts = cat_counts.sort_values('flagged_pct', ascending=False)

print('Failing questions by INCOSE category:')
display(cat_counts)

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Absolute count
axes[0].barh(cat_counts['category'], cat_counts['flagged'], color='tomato', alpha=0.7)
axes[0].set_xlabel('Number of Flagged Questions')
axes[0].set_title(f'Flagged Questions by Category (n={len(flagged_qids)})')
axes[0].invert_yaxis()

# Percentage of category
axes[1].barh(cat_counts['category'], cat_counts['flagged_pct'], color='darkorange', alpha=0.7)
axes[1].set_xlabel('% of Category Questions Flagged')
axes[1].set_title('Flagged as % of Category')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig(output_dir / 'fig_flagged_by_category.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: fig_flagged_by_category.png')

---
## 7. Summary & Export

### 7.1 Consolidated Flagged Questions Table

In [ ]:
# Build the master flagged-questions DataFrame
# Start with all questions and their failure rates
flagged_master = question_stats[['question_id', 'failure_rate', 'n_models',
                                  'category', 'correct_answer', 'question']].copy()

# Add position-invariant failure info
flagged_master = flagged_master.merge(
    pos_invariant[['question_id', 'pct_all_wrong']].rename(columns={'pct_all_wrong': 'position_invariant_fail_rate'}),
    on='question_id', how='left'
)

# Add consensus wrong answer info
flagged_master = flagged_master.merge(
    consensus_df[['question_id', 'consensus_wrong_answer', 'consensus_rate']],
    on='question_id', how='left'
)

# Add OSQ mean score
flagged_master = flagged_master.merge(
    osq_question_stats[['question_id', 'mean_total_score']].rename(
        columns={'mean_total_score': 'mean_osq_score'}
    ),
    on='question_id', how='left'
)

# Add discrimination index
flagged_master = flagged_master.merge(
    disc_df[['question_id', 'discrimination', 'difficulty']],
    on='question_id', how='left'
)

# Determine flag reasons
def get_flag_reasons(row):
    reasons = []
    if row['failure_rate'] >= FAILURE_RATE_THRESHOLD:
        reasons.append('high_failure_rate')
    if pd.notna(row.get('consensus_rate')) and row['consensus_rate'] >= CONSENSUS_WRONG_THRESHOLD:
        reasons.append('consensus_wrong_answer')
    if pd.notna(row.get('mean_osq_score')) and row['mean_osq_score'] < LOW_OSQ_SCORE_THRESHOLD:
        reasons.append('low_osq_score')
    if pd.notna(row.get('discrimination')) and row['discrimination'] < LOW_DISCRIMINATION_THRESHOLD:
        reasons.append('low_discrimination')
    if pd.notna(row.get('discrimination')) and row['discrimination'] < 0:
        reasons.append('negative_discrimination')
    return '; '.join(reasons) if reasons else ''

flagged_master['flag_reasons'] = flagged_master.apply(get_flag_reasons, axis=1)
flagged_master['n_flags'] = flagged_master['flag_reasons'].apply(
    lambda x: len(x.split('; ')) if x else 0
)

# Filter to only flagged questions
flagged_only = flagged_master[flagged_master['n_flags'] > 0].sort_values(
    ['n_flags', 'failure_rate'], ascending=[False, False]
).reset_index(drop=True)

print(f'=== FLAGGED QUESTIONS SUMMARY ===')
print(f'Total questions analyzed: {len(flagged_master)}')
print(f'Total questions flagged: {len(flagged_only)}')
print(f'\nFlag breakdown:')
print(f'  High failure rate (>= {FAILURE_RATE_THRESHOLD}): {(flagged_master["failure_rate"] >= FAILURE_RATE_THRESHOLD).sum()}')
print(f'  Consensus wrong answer (>= {CONSENSUS_WRONG_THRESHOLD}): {(flagged_master["consensus_rate"].fillna(0) >= CONSENSUS_WRONG_THRESHOLD).sum()}')
print(f'  Low OSQ score (< {LOW_OSQ_SCORE_THRESHOLD}): {(flagged_master["mean_osq_score"].fillna(100) < LOW_OSQ_SCORE_THRESHOLD).sum()}')
print(f'  Low discrimination (< {LOW_DISCRIMINATION_THRESHOLD}): {(flagged_master["discrimination"].fillna(1) < LOW_DISCRIMINATION_THRESHOLD).sum()}')
print(f'  Negative discrimination: {(flagged_master["discrimination"].fillna(0) < 0).sum()}')

print(f'\nQuestions with multiple flags:')
for nf in sorted(flagged_only['n_flags'].unique(), reverse=True):
    count = (flagged_only['n_flags'] == nf).sum()
    print(f'  {nf} flags: {count} questions')

### 7.2 Export Flagged Questions

In [ ]:
# Export to CSV
csv_path = output_dir / 'table_flagged_questions.csv'
flagged_only.to_csv(csv_path, index=False)
print(f'Exported: {csv_path}')

# Also export the full question stats for reference
full_csv_path = output_dir / 'table_all_question_stats.csv'
flagged_master.to_csv(full_csv_path, index=False)
print(f'Exported: {full_csv_path}')

# Export a LaTeX summary table (top flagged questions)
latex_df = flagged_only.head(30)[[
    'question_id', 'failure_rate', 'consensus_rate',
    'mean_osq_score', 'discrimination', 'n_flags', 'flag_reasons'
]].copy()

latex_df.columns = ['Q ID', 'Fail Rate', 'Consensus', 'OSQ Score', 'Discrim.', 'Flags', 'Reasons']

latex_str = latex_df.to_latex(
    index=False,
    float_format='%.3f',
    na_rep='--',
    caption='Top flagged questions for manual review',
    label='tab:flagged-questions'
)

latex_path = output_dir / 'table_flagged_questions.tex'
with open(latex_path, 'w') as f:
    f.write(latex_str)
print(f'Exported: {latex_path}')

### 7.3 Summary Statistics

In [ ]:
print('=' * 80)
print('QUESTION DIFFICULTY ANALYSIS - FINAL SUMMARY')
print('=' * 80)

print(f'\nDataset:')
print(f'  MCQ questions analyzed: {len(question_stats)}')
print(f'  OSQ questions analyzed: {len(osq_question_stats)}')
print(f'  Models: {mcq_df["model"].nunique()}')

print(f'\nFlagging Results:')
print(f'  Total unique questions flagged: {len(flagged_only)}')
print(f'  Percentage of questions flagged: {len(flagged_only)/len(flagged_master)*100:.1f}%')

print(f'\nBy category (flagged / total):')
for _, row in cat_counts.iterrows():
    if row['flagged'] > 0:
        print(f'  {row["category"]}: {int(row["flagged"])} / {int(row["total"])} ({row["flagged_pct"]:.1f}%)')

print(f'\nExported files:')
print(f'  {csv_path}')
print(f'  {full_csv_path}')
print(f'  {latex_path}')
print(f'  {output_dir}/fig_failure_rate_distribution.png')
print(f'  {output_dir}/fig_hardest_questions_heatmap.png')
print(f'  {output_dir}/fig_osq_question_scores.png')
print(f'  {output_dir}/fig_mcq_vs_osq_question.png')
print(f'  {output_dir}/fig_difficulty_vs_discrimination.png')
print(f'  {output_dir}/fig_flagged_by_category.png')

print(f'\nRecommendation: Review the {len(flagged_only)} flagged questions manually.')
print(f'Priority: Questions with multiple flags (especially consensus_wrong_answer + high_failure_rate).')
print('=' * 80)